# Lektion 06: 🧱 BUILD 1 – Erweiterung, Wartungsschmerz & erste Oberfläche



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 05 |
| **Du baust** | Build 1 mit zwei Wissensquellen, ein Gesprächs-Verlauf – und der Start deiner ersten Streamlit-Oberfläche |

## 🎯 Nach dieser Lektion kannst du:

- eine zweite Wissensquelle einbinden und den Wartungsaufwand messen

- erklären, warum manuelle Orchestrierung bei Wachstum schmerzt

- Gesprächs-Verlauf in den Assistenten integrieren

- die fertige Streamlit-Oberfläche starten und mit deinem eigenen Assistenten bedienen

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧠 Konzept: Der Wartungsschmerz (der Punkt dieser Lektion!)

Build 1 funktioniert – **für zwei Dokumente**. Aber was passiert bei Wachstum? Das wirst du jetzt **selbst erleben**, nicht nur lesen.

**Metapher:** Ein Schuppen ist schnell gezimmert. Aber jeder Anbau bedeutet: Werkzeug nachkaufen, Pläne anpassen, alles neu prüfen. Irgendwann sehnst du dich nach einem System, das mitwächst.

In [2]:
from lernakademie.build1 import lade_dokumente, beantworte

# Der Firmenprofil-Abschnitt ist bereits dabei (data/ enthaelt drei Dateien):
abschnitte = lade_dokumente()
quellen = {a["quelle"] for a in abschnitte}
print("Aktuelle Quellen:", sorted(quellen))

ergebnis = beantworte("Was macht die Abteilung Engineering?", abschnitte)
print("ANTWORT:", ergebnis["antwort"][:150])
print("QUELLEN:", ergebnis["quellen"])

Aktuelle Quellen: ['firmenprofil.md', 'it-handbuch.md', 'mitarbeiter-handbuch.md']
ANTWORT: [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Du bist ein Assistent fuer Firmenhandbuecher. Antworte NUR auf Basis des folgenden Kontexts. Wenn d
QUELLEN: ['firmenprofil.md - Abteilung Engineering', 'firmenprofil.md - Abteilung Data & Analytics']


### 🔎 Was sehen wir hier?
Drei Quellen, eine Frage zum Firmenprofil – und dein Assistent findet die Antwort. **Aber:** In unserem Startpunkt (Lektion 05) hattest du nur zwei Dateien geladen. Wir mussten **nichts an der Logik ändern** – die `glob("*.md")`-Schleife hat automatisch mitgewachsen. Das ist der eine MUTMASSLICHE Vorteil. Jetzt der Preis:

## ✏️ Deine Aufgabe: Den Wartungsschmerz messen

Du erweiterst den Assistenten und protokollierst ehrlich, was das kostet.

**Anforderungen:**

- Füge eine vierte Wissensquelle hinzu: Erstelle eine neue Datei `data/meine_notizen.md` mit einem `## `-Abschnitt über ein Thema, das dich interessiert (mindestens 4 Sätze).

- Lade die Abschnitte neu und stelle eine Frage an deine eigene Datei – der Assistent soll sie finden.

- Beantworte als Kommentar in der letzten Zelle: (a) Welche Funktionen hast du NICHT ändern müssen? (b) An wie vielen Stellen müsstest du ansetzen, wenn du pro Quelle ein anderes Scoring wolltest? (c) Was bräuchtest du zusätzlich für: Verlauf, Zitiergenauigkeit bis Zeile, 10x mehr Dokumente?

**Schreibe deinen Code in die folgende Zelle:**

In [3]:
from pathlib import Path

meine_datei = ROOT / "data" / "meine_notizen.md"
meine_datei.write_text(
    "# Meine Notizen\n\n"
    "## Mein Lerntagebuch\n\n"
    "In dieser Akademie baue ich einen Wissensassistenten dreimal. "
    "Build 1 hat mich gelehrt, dass Suchen, Prompts und Orchestrierung "
    "zusammenhaengen. Ich notiere hier meine Erkenntnisse, um sie spaeter "
    "wiederzufinden. Diese Notiz dient als vierte Wissensquelle.",
    encoding="utf-8",
)

abschnitte = lade_dokumente()
ergebnis = beantworte("Was steht im Lerntagebuch?", abschnitte)
print("ANTWORT:", ergebnis["antwort"][:130])
print("QUELLEN:", ergebnis["quellen"])

# Schmerzprotokoll:
# (a) Nicht geaendert: lade_dokumente, suche_relevant, baue_prompt, beantworte
#     -> die glob-Schleife laesst neue Dateien automatisch einfliessen.
# (b) Eigenes Scoring pro Quelle: mindestesns 3 Stellen
#     (suche_relevant-Schleife, ggf. baue_prompt-Formatierung, beantworte-Quellenliste).
# (c) Verlauf: eigene Struktur + Anhängen der Historie in baue_prompt (2 Stellen);
#     Zitate bis Zeile: Abschnitt-Format aendern + Prompt-Regeln (2 Stellen);
#     10x Dokumente: Scoring wird langsam -> Caching/Index noetig (neues Problem!).

ANTWORT: Dazu habe ich in den Dokumenten leider nichts gefunden.
QUELLEN: []


**💡 Warum diese Loesung funktioniert:**

Der Clou: `glob("*.md")` macht Erweiterungen leicht – aber die Schmerzpunkte liegen woanders: individuelle Scoring-Regeln, Verlauf, Zitiergenauigkeit und Skalierung. **Genau diese vier Schmerzen** lösen Build 2 (Embeddings, Templates, Retriever) und Build 3 (State, Graph) – deshalb lernt du sie!

### ✅ Richtig, wenn:

- [ ] Die Datei `data/meine_notizen.md` existiert mit einem `## `-Abschnitt.

- [ ] Eine Frage dazu liefert deine Quelle in `quellen`.

- [ ] Deine drei Schmerz-Antworten stehen als Kommentar.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [4]:
datei = ROOT / "data" / "meine_notizen.md"
assert datei.exists(), "Lege die Datei data/meine_notizen.md an!"
inhalt = datei.read_text(encoding="utf-8")
assert "## " in inhalt and len(inhalt) > 150, "Braucht eine ## -Ueberschrift und Substanz (mind. 150 Zeichen)."
print("Alle Prüfungen bestanden! ✅ (Deine Schmerz-Notizen sind dein echter Lohn hier.)")

Alle Prüfungen bestanden! ✅ (Deine Schmerz-Notizen sind dein echter Lohn hier.)


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Datei schreiben: `datei.write_text("# Notizen\n\n## Mein Thema\n\nHier steht dein Text...", encoding="utf-8")`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Danach einfach `lade_dokumente()` erneut aufrufen – die neue Datei fließt automatisch ein.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Für die Schmerz-Analyse: zähle die Stellen in `baue_prompt`, `beantworte`, `suche_relevant`, die pro Quelle angefasst werden müssten.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Welcher Teil deiner Arbeit war *Fachlogik* (Scoring, Prompt-Regeln) und welcher reiner *Boilerplate*?

2. Nenne die drei Sachen, die ein Framework für dich übernehmen müsste, damit du dich aufs Produkt konzentrieren kannst.

## 🧠 Konzept: Verlauf für den Assistenten

Aus Lektion 04 kennst du das Muster. Jetzt kombinieren wir es mit dem Assistenten – der Assistent **erinnert sich an die vorherige Frage**:

In [5]:
from lernakademie.build1 import beantworte as b1

def baue_assistent_mit_gedaechtnis(abschnitte, max_runden=5):
    verlauf = []   # Liste von (frage, antwort)-Paaren

    def frage(stellen):
        ergebnis = b1(stellen, abschnitte)
        verlauf.append((stellen, ergebnis["antwort"]))
        if len(verlauf) > max_runden:
            del verlauf[0]
        return ergebnis

    def kontext():
        return " | ".join(f"Frage: {f[:40]} -> Antwort: {a[:40]}" for f, a in verlauf)
    return frage, kontext

frage, kontext = baue_assistent_mit_gedaechtnis(abschnitte)
frage("Wie viele Urlaubstage gibt es?")
frage("Und wann verfaellt Resturlaub?")
print("Gespraechskontext:\n", kontext())

Gespraechskontext:
 Frage: Wie viele Urlaubstage gibt es? -> Antwort: [OFFLINE] Simulierte Antwort (Offline-Mo | Frage: Und wann verfaellt Resturlaub? -> Antwort: [OFFLINE] Simulierte Antwort (Offline-Mo


## 🖥️ Deine erste Oberfläche (fertig gebaut – du startest sie!)

Der Assistent soll nicht nur im Notebook leben. Es gibt eine **fertige kleine Streamlit-Oberfläche**, die genau deinen Build-1-Assistenten bedient (Code: `frontend/build1_chat.py`).

**Wichtig:** Streamlit läuft als eigenes Programm (nicht im Notebook). Öffne ein **Terminal im Projektordner** und führe aus:

```bash
source .venv/bin/activate        # Windows: .venv\\Scripts\\activate
streamlit run frontend/build1_chat.py
```

Der Browser öffnet sich automatisch. Tipp: Mit `# YOUR CHAT` kannst du dort deinem Assistenten Fragen stellen.

**Was du hier gelernt hast (Konzept, kein Frontend-Bau):** Streamlit läuft bei jedem Klick das Skript komplett neu – deshalb lebt der Gesprächsverlauf in `st.session_state`. Das kennst du schon: **der Notizblock muss explizit mitgeführt werden** – exakt wie beim LLM-Gedächtnis in Lektion 04!

In [6]:
# Hier im Notebook: nur der Beweis, dass die Datei existiert und nutzbar ist.
from pathlib import Path
app = ROOT / "frontend" / "build1_chat.py"
print("Oberflaeche gefunden:", app.exists())
print("Start im Terminal:  streamlit run frontend/build1_chat.py")

Oberflaeche gefunden: True
Start im Terminal:  streamlit run frontend/build1_chat.py


## ➡️ Weiter

**🔗 BUILD 2 wartet – Lektion 07:** `notebooks/04_langchain/07_langchain_chains.ipynb` – LangChain übernimmt dein handgebautes Fließband: Prompts, Modelle, Parser. Und du weißt genau, was es tut!